# Pool Ablation Benchmark Notebook

Bu notebook, `benchmark_pipeline_pool_ablation.py` dosyasını kullanarak havuz ablasyon çalışmasını çalıştırır ve iki Excel çıktısı üretir:

- `benchmark_results_pool_ablation.xlsx`
- `benchmark_summary_pool_ablation.xlsx`

Not: Gerekirse `PROJECT_DIR` ve `DATASET_ROOT` yollarını kendi bilgisayarınıza göre güncelleyin.


In [1]:
import os
import sys
from pathlib import Path

# benchmark_pipeline_pool_ablation_corrected.py dosyasının bulunduğu klasör
PROJECT_DIR = Path.cwd()
# Örnek: PROJECT_DIR = Path(r"C:\\Users\\Ersin\\Desktop\\claude_smote_project")

if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

from benchmark_pipeline_pool_ablation import (
    run_pipeline,
    compute_dataset_means,
    compute_classifier_means,
    build_summary_table,
    build_overall_method_summary,
    save_outputs,
)

print(f"PROJECT_DIR: {PROJECT_DIR}")


PROJECT_DIR: C:\Users\Ersin\Desktop\claude_smote_project -II


In [2]:
# Veri klasörü
DATASET_ROOT = Path("./datasets_keel")
# Örnek: DATASET_ROOT = Path(r"C:\\Users\\Ersin\\Desktop\\datasets_keel")

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset klasörü bulunamadı: {DATASET_ROOT.resolve()}")

datasets = sorted([p.name for p in DATASET_ROOT.iterdir() if p.is_dir()])
print(f"{len(datasets)} dataset bulundu.")
print(datasets[:20])


44 dataset bulundu.
['abalone19', 'abalone9-18', 'ecoli-0-1-3-7_vs_2-6', 'ecoli-0_vs_1', 'ecoli1', 'ecoli2', 'ecoli3', 'ecoli4', 'glass-0-1-2-3_vs_4-5-6', 'glass-0-1-6_vs_2', 'glass-0-1-6_vs_5', 'glass0', 'glass1', 'glass2', 'glass4', 'glass5', 'glass6', 'haberman', 'iris0', 'new-thyroid1']


In [8]:
# Ana pipeline çalıştırma
fold_df = run_pipeline(dataset_root=str(DATASET_ROOT))

print("Fold Results shape:", fold_df.shape)
fold_df.head()


Genel İlerleme:   0%|          | 0/880 [00:00<?, ?adım/s]

Toplam 44 dataset bulundu.
Toplam adım sayısı: 880

Klasik karşılaştırma seti (12 yöntem): Original, ROS, SMOTE, ADASYN, Borderline_SMOTE1, Borderline_SMOTE2, MWMOTE, Safe_Level_SMOTE, G_SMOTE, kmeans_SMOTE, SMOTEENN, SMOTETomek
Pool senaryosu sayısı: 7
Toplam değerlendirilen yöntem sayısı: 40

Dataset: abalone19

  Fold 1/5
    Train: (3339, 8), Minority: 26, Majority: 3313
    Adım 1/880 (%0.1) | Kalan: 879 | Tahmini Süre: ? | Rate:  25% (needed=822) | Model: 200 ✓ | Adım Süresi: 01:07
    Adım 2/880 (%0.2) | Kalan: 878 | Tahmini Süre: 16:19:36 | Rate:  50% (needed=1644) | Model: 200 ✓ | Adım Süresi: 01:46
    Adım 3/880 (%0.3) | Kalan: 877 | Tahmini Süre: 21:03:42 | Rate:  75% (needed=2465) | Model: 200 ✓ | Adım Süresi: 02:21
    Adım 4/880 (%0.5) | Kalan: 876 | Tahmini Süre: 25:28:37 | Rate: 100% (needed=3287) | Model: 200 ✓ | Adım Süresi: 02:55

  Fold 2/5
    Train: (3339, 8), Minority: 26, Majority: 3313
    Adım 5/880 (%0.6) | Kalan: 875 | Tahmini Süre: 29:44:30 | Rate:  25% (n

,dataset,fold,augmentation_rate,method,classifier,F1,GMean,BalancedAccuracy,ROC_AUC
0,abalone19,1,0.25,Original,RandomForest,0.0,0.0,0.500000,0.636610
1,abalone19,1,0.25,Original,SVM_RBF,0.0,0.0,0.500000,0.878166
2,abalone19,1,0.25,Original,KNN,0.0,0.0,0.500000,0.483715
3,abalone19,1,0.25,Original,LogisticRegression,0.0,0.0,0.500000,0.652191
4,abalone19,1,0.25,Original,GaussianNB,0.0,0.0,0.420989,0.635505


In [5]:
# Özet tabloları oluşturma
dataset_df = compute_dataset_means(fold_df)
classifier_df = compute_classifier_means(dataset_df)
summary_df = build_summary_table(fold_df)
overall_summary = build_overall_method_summary(summary_df)

print("Dataset Mean Results shape:", dataset_df.shape)
print("Classifier Mean Results shape:", classifier_df.shape)
print("Summary Table shape:", summary_df.shape)
print("Overall Method Summary shape:", overall_summary.shape)


Dataset Mean Results shape: (35200, 8)
Classifier Mean Results shape: (800, 7)
Summary Table shape: (3200, 9)
Overall Method Summary shape: (40, 5)


In [6]:
# Kontrol amaçlı önizleme
display(summary_df.head(20))
display(overall_summary.head(20))


,augmentation_rate,classifier,metric,method,Average,Winner_Total,Mean_Rank,Final_Rank,Total_Datasets
0,0.25,GaussianNB,F1,ADASYN,0.489391,4,21.340909,21,44
1,0.25,GaussianNB,F1,Borderline_SMOTE1,0.501845,6,16.693182,5,44
2,0.25,GaussianNB,F1,Borderline_SMOTE2,0.469619,2,23.897727,36,44
3,0.25,GaussianNB,F1,G_SMOTE,0.493675,1,20.886364,16,44
4,0.25,GaussianNB,F1,MWMOTE,0.482383,5,21.181818,20,44
5,0.25,GaussianNB,F1,Original,0.486279,5,21.943182,27,44
6,0.25,GaussianNB,F1,Proposed_Boundary__Pool_All,0.495156,4,19.488636,14,44
7,0.25,GaussianNB,F1,Proposed_Boundary__Pool_Core,0.499347,1,20.113636,15,44
8,0.25,GaussianNB,F1,Proposed_Boundary__Pool_Drop_kmeans,0.490177,3,21.806818,25,44
9,0.25,GaussianNB,F1,Proposed_Boundary__Pool_Drop_kmeans_Borderline,0.488474,3,22.238636,29,44


,method,Average_of_Averages,Total_Winner_Count,Mean_of_Mean_Rank,Mean_of_Final_Rank
35,SMOTE,0.763234,426,17.791619,9.9750
11,Proposed_Boundary__Pool_SMOTE_ADASYN_MWMOTE_G_...,0.759861,387,18.681108,13.6750
10,Proposed_Boundary__Pool_Drop_kmeans_Borderline...,0.759656,400,18.881392,14.6125
7,Proposed_Boundary__Pool_Core,0.759537,384,18.947585,14.9375
29,Proposed_Random__Pool_Drop_kmeans,0.760026,378,18.987216,15.2000
27,Proposed_Random__Pool_All,0.759357,383,19.016903,15.3875
8,Proposed_Boundary__Pool_Drop_kmeans,0.759535,416,19.031534,15.7125
33,Proposed_Random__Pool_Strong_Classics,0.759768,387,19.178551,16.0000
28,Proposed_Random__Pool_Core,0.759304,372,19.198580,16.1500
12,Proposed_Boundary__Pool_Strong_Classics,0.759168,415,19.216477,16.5375


In [7]:
# Excel çıktılarını kaydetme
save_outputs(
    fold_df,
    dataset_df,
    classifier_df,
    summary_df,
)

print("Tamamlandı!")
print("Kaydedilen dosyalar:")
print("- benchmark_results_pool_ablation.xlsx")
print("- benchmark_summary_pool_ablation.xlsx")



✅ Detaylı sonuç dosyası kaydedildi: benchmark_results_pool_ablation.xlsx
✅ Özet tablo dosyası kaydedildi: benchmark_summary_pool_ablation.xlsx
Tamamlandı!
Kaydedilen dosyalar:
- benchmark_results_pool_ablation.xlsx
- benchmark_summary_pool_ablation.xlsx
